<a href="https://colab.research.google.com/github/666fl/my-portfolio/blob/main/Licence_Plate_Recog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import all the dependencies

In [ ]:
!pip install easyocr

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import easyocr
import imutils

# Load and Display Original Image

In [ ]:
img = cv2.imread('image1.jpg')
if img is None:
    raise ValueError("Error loading image - ensure 'image1.jpg' exists in the current directory")
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title('Original Image')
plt.show()

# Convert to Grayscale

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
plt.imshow(gray, cmap='gray')
plt.title('Grayscale Image')
plt.show()

# Apply Bilateral Filter and Edge Detection

In [ ]:
filtered = cv2.bilateralFilter(gray, 10, 17, 17)  # noise reduction
edged = cv2.Canny(filtered, 30, 200)  # edge detection using Canny's algorithm
plt.imshow(edged, cmap='gray')
plt.title('Edge Detection')
plt.show()

# Find Contours and Apply Mask

In [ ]:
keypoints = cv2.findContours(edged.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
contours = imutils.grab_contours(keypoints)
contours = sorted(contours, key=cv2.contourArea, reverse=True)[:10]

In [ ]:
location = None
for contour in contours:
    approx = cv2.approxPolyDP(contour, 10, True)
    if len(approx) == 4:
        location = approx
        break

if location is None:
    print("Warning: No 4-sided contour found")

In [ ]:
print("License plate location (corner coordinates):")
print(location)

# Extract License Plate Region

In [ ]:
if location is not None:
    mask = np.zeros(gray.shape, np.uint8)
    new_image = cv2.drawContours(mask, [location], 0, 255, -1)
    new_image = cv2.bitwise_and(img, img, mask=mask)

In [ ]:
if location is not None:
    plt.imshow(cv2.cvtColor(new_image, cv2.COLOR_BGR2RGB))
    plt.title('Extracted License Plate Region')
    plt.show()

# Perform OCR Recognition

In [ ]:
# Initialize EasyOCR reader
reader = easyocr.Reader(['en'])

# Perform OCR on the extracted license plate region
if location is not None:
    result = reader.readtext(new_image)
    if result:
        text = result[0][1]
    else:
        text = "No text found"
        print("Warning: OCR could not detect any text")
else:
    text = "No license plate found"
    print("Warning: License plate region not found")

In [ ]:
print(f"Detected License Plate Text: {text}")

# Draw Results on Original Image

In [ ]:
if location is not None and text != "No license plate found":
    # Create a copy of the original image
    result_img = img.copy()
    
    # Draw rectangle around the license plate
    result_img = cv2.rectangle(result_img, tuple(location[0][0]), tuple(location[2][0]), (0, 255, 0), 3)
    
    # Add text with the detected license plate number
    font = cv2.FONT_HERSHEY_SIMPLEX
    result_img = cv2.putText(result_img, text, org=(location[0][0][0], location[1][0][1] + 60), 
                              fontFace=font, fontScale=1, color=(0, 255, 0), thickness=2, lineType=cv2.LINE_AA)
    
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.title(f'License Plate: {text}')
    plt.show()
else:
    print("Cannot draw results - no valid license plate detected")